## Cache behaviours, TTL & cache keys

A **cache behaviour** is a routing rule on a distribution: a **path pattern**, the **origin** it maps to, and the caching settings for that path. A distribution has a default behaviour (`*`) plus any number of more specific ones, evaluated in **precedence order**. The classic split: `/api/*` → ALB with **TTL 0** (dynamic, don't cache), `/images/*` → S3 with a **1-day** TTL, `*` → S3 with a **1-hour** default. Per behaviour you also set the **viewer protocol policy**, allowed **HTTP methods**, attached **edge functions**, and the **cache policy**.

**TTL** governs freshness. CloudFront honours the origin's `Cache-Control`/`Expires` headers, falling back to the behaviour's defaults — **default 24 h, min 0, max effectively unbounded**. A minimum TTL of 0 still caches at the edge; it just forces a **revalidation** with the origin on each request.

The **cache key** is what CloudFront hashes to decide "same object or not" — by default just the host + path, but you can add specific **query strings, headers, and cookies**. This is the subtle cost lever: **narrow the key** (strip irrelevant query params) and your hit ratio climbs; **widen it** carelessly (include a per-user cookie) and every request becomes a unique object — a cache that never hits. **Invalidations** force-expire cached objects (e.g. `/*` after a deploy); the first 1,000 paths/month are free.